# CLAUDE-D01 — shot-00 free-GPU PoC (Wan2.1-T2V-1.3B)

Generates **shot-00 only** of CLAUDE-D01 with the open-weight **Wan2.1-T2V-1.3B** model
(Apache 2.0 license - commercial use permitted) on a **free GPU runtime**
(Google Colab free T4, or Kaggle Notebooks free T4/P100 - both have >=16GB VRAM,
comfortably above this model's ~8.2GB requirement).

**The prompt below is copied verbatim from `production/jobs/CLAUDE-D01.json`'s
shot-00 - the creative has not been changed.**

## How to run
1. **Colab**: `Runtime` -> `Change runtime type` -> `T4 GPU` -> Save. Then `Runtime` -> `Run all`.
2. **Kaggle**: open as a Notebook, in the right sidebar set `Accelerator` -> `GPU T4 x2` (or P100). Then `Run All`.
3. Wait for the last cell to finish (first run downloads ~3GB of weights, then generates).
4. Download `shot-00_wan21.mp4` from the file browser on the left.
5. On your own machine, ingest it into the pipeline:
   ```bash
   python -m production.cli ingest-shot CLAUDE-D01 shot-00 /path/to/shot-00_wan21.mp4 \
       --service wan21_colab_free_gpu --commercial-clear \
       --license-note "Wan2.1-T2V-1.3B is Apache 2.0 (commercial use permitted); generated on a free Colab/Kaggle GPU runtime, whose own ToS restricts abuse of the compute service, not commercial use of your own output."
   ```

This was prepared and code-checked (import/class/signature verified against the
actual installed `diffusers` library) without a GPU available in that checking
environment, so the actual generation has **not** been run end-to-end yet -
this notebook run is that first real execution.

In [ ]:
!pip install -q -U diffusers accelerate transformers imageio imageio-ffmpeg
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
import torch
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.bfloat16)
pipe.to("cuda")

In [ ]:
# Verbatim from production/jobs/CLAUDE-D01.json, shot-00. Creative unchanged.
prompt = "明るいカフェの窓際で、驚いた表情でスマホの画面を見つめる20代女性。自然光、縦構図9:16、手元にスマホ、清潔感のあるカジュアルな服装。"
negative_prompt = "blurry, distorted hands, extra fingers, low quality, text, watermark, logo"

# 9:16 vertical (swap of the model's documented 480x832 landscape default).
# num_frames must satisfy (num_frames - 1) % 4 == 0; 81 frames @ 16fps ~= 5.06s,
# close to shot-00's declared 6s - the pipeline's ingest-shot trims/pads exactly
# to the job's declared duration on ingestion, so this doesn't need to match exactly.
output = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    height=832,
    width=480,
    num_frames=81,
    num_inference_steps=30,   # reduced from the default 50 for free-GPU time budget
    guidance_scale=5.0,
)

export_to_video(output.frames[0], "shot-00_wan21.mp4", fps=16)
print("Saved shot-00_wan21.mp4 - download it from the file browser.")